# Satellite change detection with VLM interpretation

Run the cells top to bottom. The module cells write the pipeline files to the working folder; the stage cells then run them on real Sentinel-2 imagery from AWS. Edit `config.yaml` (area, date windows) first. Set `ANTHROPIC_API_KEY` before the interpretation stage.

In [ ]:
%pip install -q numpy pandas pyyaml opencv-python-headless scikit-image scipy rasterio shapely pyproj pystac-client anthropic torch

### `config.yaml`

In [ ]:
%%writefile config.yaml
paths:
  work: runs
  catalog: runs/catalog.csv
  pairs: runs/pairs.csv
  prepared: runs/prepared
  reports: runs/reports

aoi:
  bbox: [73.03, 18.96, 73.09, 19.02]
  datetime: "2021-01-01/2026-03-31"

sources:
  - name: sentinel2
    type: api
    url: https://earth-search.aws.element84.com/v1
    collections: [sentinel-2-l2a]
    modality: optical
    query: {"eo:cloud_cover": {lt: 15}}
    assets: [red, green, blue, nir]
    mask_asset: scl
    mask_classes: [3, 8, 9, 10]
    gsd: 10
    max_items: 300
# Umbra static catalog is large to traverse; enable only when needed
#   - name: umbra
#     type: static
#     url: https://s3.us-west-2.amazonaws.com/umbra-open-data-catalog/stac/catalog.json
#     modality: sar
#     assets: [GEC]
#     gsd: 0.5
#     max_items: 300
#     max_scan: 20000

pairing:
  min_gap_days: 30
  max_gap_days: 2500
  target_gap_days: 365
  windows:
    before: "2021-01-01/2021-03-31"
    after: "2026-01-01/2026-03-31"
  min_aoi_coverage: 0.8
  max_gsd_ratio: 1.25
  max_off_nadir_diff: 10
  same_platform: false
  same_orbit_state: true
  max_cloud: 15
  max_pairs_per_source: 3
  strict_missing: false
  target_res_m: null
  max_pixels: 36000000
  nodata_value: 0
  min_valid_fraction: 0.7
  reg_max_shift_px: 1.5
  reg_repair_limit_px: 8
  reg_min_response: 0.05
  pct_low: 2
  pct_high: 98
  match_histograms: true
  sar_input: amplitude
  dedupe_cell_deg: 0.05

detect:
  method_by_modality: {optical: classical, sar: classical}
  bands: [0, 1, 2]
  blur_sigma: {optical: 1.5, sar: 3.0}
  threshold: {type: robust_z, value: 3.5}
  morph_kernel: 3
  min_area_px: 30
  learned:
    checkpoint: runs/siamunet.pt
    device: cpu
    tile: 256
    overlap: 32
    threshold: 0.5

model:
  in_channels: 3
  widths: [32, 64, 128, 256]

train:
  root: data/LEVIR-CD
  dirs: {a: A, b: B, label: label}
  splits: {train: train, val: val}
  patch: 256
  batch: 8
  epochs: 40
  lr: 0.001
  device: cpu
  workers: 2
  seed: 0
  pos_weight: 5.0
  dice_weight: 1.0
  out: runs/siamunet.pt

vlm:
  model: claude-sonnet-5-5
  judge_model: claude-sonnet-5-5
  max_tokens: 700
  chip_px: 384
  context_pad: 0.6
  min_chip_px: 48
  jpeg_quality: 90
  rgb_idx: [0, 1, 2]
  display_pct: [2, 98]

interpret:
  max_regions: 20
  swap_rate: 1.0
  null_rate: 0.5
  judge: true
  seed: 0
  ring_px: 12
  min_prob: 0.5
  min_contrast: 0.2
  reject_on: [null_hallucination, swap_inconsistent, judge_rejected, weak_pixel_evidence]
  summarize: true
  change_types: [new_construction, demolition, road_change, vegetation_gain, vegetation_loss, water_change, vehicle_or_object_movement, ground_disturbance, other]
  nuisance_types: [none, shadow, seasonal_vegetation, cloud, illumination, registration_error, sensor_artifact]
  inverse_types:
    new_construction: demolition
    demolition: new_construction
    vegetation_gain: vegetation_loss
    vegetation_loss: vegetation_gain
    road_change: road_change
    water_change: water_change
    vehicle_or_object_movement: vehicle_or_object_movement
    ground_disturbance: ground_disturbance
    other: other

prompts:
  system: >
    You are an imagery analyst comparing two co-registered satellite images of the same location.
    Report only changes visible in the images. Do not speculate about purpose or ownership.
    If the difference is explainable by illumination, shadow, season, cloud or sensor effects, say so.
    Answer with one JSON object and nothing else.
  describe: >
    Image 1 is BEFORE ({before_date}), image 2 is AFTER ({after_date}), image 3 is AFTER with the
    candidate region outlined. Modality: {modality}. Pixel size about {gsd} m.
    Return JSON with keys: change_detected (bool), change_type (one of {types}),
    object (short noun phrase), before_state, after_state, description (one sentence),
    nuisance (one of {nuisance}), confidence (0 to 1).
  judge: >
    Image 1 is BEFORE ({before_date}), image 2 is AFTER ({after_date}), image 3 is AFTER with the
    region outlined. A candidate claim: {claim}
    Decide whether the images support the claim. Return JSON with keys: supported (bool), reason (one sentence).
  summary: >
    Summarize these verified and uncertain changes between {before_date} and {after_date} in at most five
    sentences, plainly and without speculation. Findings: {findings}

export:
  out_dir: data/real
  preview_px: 1024
  files: [before.tif, after.tif, valid.tif, prob.tif, pair.json, regions.json]

### `common.py`

In [ ]:
%%writefile common.py
import json
import logging
from pathlib import Path
import yaml


def load_config(path):
    with open(path) as f:
        return yaml.safe_load(f)


def ensure(path):
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


def utm_epsg(lon, lat):
    zone = int((lon + 180) // 6) + 1
    return (32600 if lat >= 0 else 32700) + zone


def get_logger(name):
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")
    return logging.getLogger(name)


def dump_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)


def load_json(path):
    with open(path) as f:
        return json.load(f)

### `discover.py`

In [ ]:
%%writefile discover.py
import json
import pandas as pd
import pystac
from pystac_client import Client
from shapely.geometry import box, shape
from common import ensure, get_logger

log = get_logger("discover")


def normalize(item, src):
    p = item.properties
    assets = {k: item.assets[k].href for k in src.get("assets", []) if k in item.assets}
    if src.get("mask_asset") and src["mask_asset"] in item.assets:
        assets["_mask"] = item.assets[src["mask_asset"]].href
    return {
        "id": item.id,
        "source": src["name"],
        "modality": src["modality"],
        "collection": item.collection_id,
        "datetime": p.get("datetime") or p.get("start_datetime"),
        "platform": p.get("platform"),
        "gsd": p.get("gsd", src.get("gsd")),
        "cloud": p.get("eo:cloud_cover"),
        "off_nadir": p.get("view:off_nadir"),
        "sun_elev": p.get("view:sun_elevation"),
        "orbit_state": p.get("sat:orbit_state"),
        "license": p.get("license"),
        "geometry": json.dumps(item.geometry),
        "assets": json.dumps(assets),
        "needed": len(assets) >= len(src.get("assets", [])),
    }


def search_api(src, cfg):
    client = Client.open(src["url"])
    res = client.search(
        collections=src["collections"],
        bbox=cfg["aoi"]["bbox"],
        datetime=cfg["aoi"]["datetime"],
        query=src.get("query"),
        max_items=src["max_items"],
    )
    return list(res.items())


def search_static(src, cfg):
    cat = pystac.Catalog.from_file(src["url"])
    aoi = box(*cfg["aoi"]["bbox"])
    start, end = (pd.to_datetime(x, utc=True) for x in cfg["aoi"]["datetime"].split("/"))
    out, scanned = [], 0
    for item in cat.get_items(recursive=True):
        scanned += 1
        if scanned > src["max_scan"] or len(out) >= src["max_items"]:
            break
        if item.geometry is None or not shape(item.geometry).intersects(aoi):
            continue
        dt = item.datetime or (pd.to_datetime(item.properties.get("start_datetime"), utc=True).to_pydatetime())
        if dt is None or not (start <= pd.Timestamp(dt).tz_convert("UTC") <= end):
            continue
        out.append(item)
    return out


def run(cfg):
    rows = []
    for src in cfg["sources"]:
        fn = search_api if src["type"] == "api" else search_static
        try:
            items = fn(src, cfg)
        except Exception as e:
            log.error("source %s failed: %s", src["name"], e)
            continue
        log.info("%s: %d items", src["name"], len(items))
        rows += [normalize(i, src) for i in items]
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df[df["needed"]].drop(columns="needed").drop_duplicates(["source", "id"])
    ensure(cfg["paths"]["work"])
    df.to_csv(cfg["paths"]["catalog"], index=False)
    log.info("catalog: %d items -> %s", len(df), cfg["paths"]["catalog"])
    return df

### `pairing.py`

In [ ]:
%%writefile pairing.py
import hashlib
import itertools
import json
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from rasterio.vrt import WarpedVRT
from rasterio.warp import transform_bounds
from shapely.geometry import box, shape
from skimage.exposure import match_histograms
from common import dump_json, ensure, get_logger, utm_epsg

log = get_logger("pairing")


def both(a, b):
    return not (a is None or b is None or pd.isna(a) or pd.isna(b))


def build_candidates(cat, cfg):
    pc = cfg["pairing"]
    aoi = box(*cfg["aoi"]["bbox"])
    cat = cat.copy()
    cat["dt"] = pd.to_datetime(cat["datetime"], utc=True, errors="coerce")
    cat = cat.dropna(subset=["dt"]).sort_values("dt")
    rows = []
    for src, g in cat.groupby("source"):
        recs = g.to_dict("records")
        geoms = [shape(json.loads(r["geometry"])).intersection(aoi) for r in recs]
        found = []
        idx = range(len(recs))
        win = pc.get("windows")
        if win:
            wa, wb = (tuple(pd.to_datetime(x, utc=True) for x in win[k].split("/")) for k in ("before", "after"))
            combos = itertools.product([i for i in idx if wa[0] <= recs[i]["dt"] <= wa[1]],
                                       [i for i in idx if wb[0] <= recs[i]["dt"] <= wb[1]])
        else:
            combos = itertools.combinations(idx, 2)
        for i, j in combos:
            a, b = recs[i], recs[j]
            gap = (b["dt"] - a["dt"]).total_seconds() / 86400
            if not pc["min_gap_days"] <= gap <= pc["max_gap_days"]:
                continue
            cov = geoms[i].intersection(geoms[j]).area / aoi.area
            if cov < pc["min_aoi_coverage"]:
                continue
            fails = []
            ga, gb = a["gsd"], b["gsd"]
            if both(ga, gb):
                if max(ga, gb) / min(ga, gb) > pc["max_gsd_ratio"]:
                    fails.append("gsd_ratio")
            elif pc["strict_missing"]:
                fails.append("gsd_missing")
            if both(a["off_nadir"], b["off_nadir"]):
                if abs(a["off_nadir"] - b["off_nadir"]) > pc["max_off_nadir_diff"]:
                    fails.append("off_nadir")
            elif pc["strict_missing"]:
                fails.append("off_nadir_missing")
            if pc["same_platform"] and a["platform"] != b["platform"] and not (pd.isna(a["platform"]) and pd.isna(b["platform"])):
                fails.append("platform")
            if pc["same_orbit_state"] and a["modality"] == "sar" and a["orbit_state"] != b["orbit_state"]:
                fails.append("orbit_state")
            for r in (a, b):
                if r["modality"] == "optical" and not pd.isna(r["cloud"]) and r["cloud"] > pc["max_cloud"]:
                    fails.append("cloud")
                    break
            if not fails:
                cl = sum(0 if pd.isna(r["cloud"]) else r["cloud"] for r in (a, b))
                found.append((cov, -cl if win else -abs(gap - pc["target_gap_days"]), gap, a, b))
        found.sort(key=lambda x: (x[0], x[1]), reverse=True)
        for cov, _, gap, a, b in found[: pc["max_pairs_per_source"]]:
            rows.append({"a": a, "b": b, "coverage": cov, "gap_days": gap})
    return rows


def grid_for(cfg, res):
    w, s, e, n = cfg["aoi"]["bbox"]
    epsg = utm_epsg((w + e) / 2, (s + n) / 2)
    minx, miny, maxx, maxy = transform_bounds("EPSG:4326", f"EPSG:{epsg}", w, s, e, n)
    width = int(np.ceil((maxx - minx) / res))
    height = int(np.ceil((maxy - miny) / res))
    return f"EPSG:{epsg}", from_origin(minx, maxy, res, res), width, height


def read_band(href, crs, transform, width, height, resampling, nodata):
    with rasterio.open(href) as src:
        with WarpedVRT(src, crs=crs, transform=transform, width=width, height=height,
                       resampling=resampling, src_nodata=src.nodata, nodata=nodata) as v:
            return v.read(1).astype("float32")


def load_scene(rec, crs, transform, w, h, pc):
    assets = json.loads(rec["assets"])
    keys = [k for k in assets if k != "_mask"]
    bands = np.stack([read_band(assets[k], crs, transform, w, h, Resampling.bilinear, pc["nodata_value"]) for k in keys])
    valid = (bands != pc["nodata_value"]).all(0)
    if "_mask" in assets:
        classes = rec.get("mask_classes") or []
        m = read_band(assets["_mask"], crs, transform, w, h, Resampling.nearest, pc["nodata_value"])
        valid &= ~np.isin(m, classes)
    return bands, valid, keys


def to_working(bands, valid, modality, pc):
    if modality == "sar":
        x = np.maximum(bands, 1e-6)
        k = 20.0 if pc["sar_input"] == "amplitude" else 10.0
        return k * np.log10(x)
    out = np.empty_like(bands)
    for i, b in enumerate(bands):
        lo, hi = np.percentile(b[valid], [pc["pct_low"], pc["pct_high"]])
        out[i] = np.clip((b - lo) / max(hi - lo, 1e-6), 0, 1)
    return out


def gray(x, valid):
    g = x.mean(0).astype("float32")
    g[~valid] = g[valid].mean() if valid.any() else 0
    return g


def phase_shift(ga, gb):
    win = cv2.createHanningWindow((ga.shape[1], ga.shape[0]), cv2.CV_32F)
    (dx, dy), resp = cv2.phaseCorrelate(ga, gb, win)
    return dx, dy, resp


def shift_image(x, dx, dy):
    m = np.float32([[1, 0, -dx], [0, 1, -dy]])
    return np.stack([cv2.warpAffine(b, m, (b.shape[1], b.shape[0]), flags=cv2.INTER_LINEAR, borderValue=0) for b in x])


def verify_and_prepare(cand, cfg):
    pc = cfg["pairing"]
    a, b = cand["a"], cand["b"]
    src = next(s for s in cfg["sources"] if s["name"] == a["source"])
    a["mask_classes"] = b["mask_classes"] = src.get("mask_classes")
    res = pc["target_res_m"] or max(a["gsd"], b["gsd"])
    crs, transform, w, h = grid_for(cfg, res)
    pid = hashlib.sha1(f"{a['id']}|{b['id']}".encode()).hexdigest()[:10]
    meta = {"pair_id": pid, "source": a["source"], "modality": a["modality"], "before_id": a["id"], "after_id": b["id"],
            "before_date": str(a["dt"]), "after_date": str(b["dt"]), "gap_days": cand["gap_days"],
            "aoi_coverage": cand["coverage"], "crs": crs, "res_m": res, "width": w, "height": h,
            "transform": list(transform)[:6], "license": [a.get("license"), b.get("license")], "steps": []}
    if w * h > pc["max_pixels"]:
        return {**meta, "verdict": "reject", "reason": "grid_too_large"}
    try:
        ba, va, keys = load_scene(a, crs, transform, w, h, pc)
        bb, vb, _ = load_scene(b, crs, transform, w, h, pc)
    except Exception as e:
        return {**meta, "verdict": "reject", "reason": f"read_failed:{e}"}
    valid = va & vb
    vf = float(valid.mean())
    meta.update(bands=keys, valid_fraction=vf)
    if vf < pc["min_valid_fraction"]:
        return {**meta, "verdict": "reject", "reason": "valid_fraction"}
    xa, xb = to_working(ba, va, a["modality"], pc), to_working(bb, vb, b["modality"], pc)
    dx, dy, resp = phase_shift(gray(xa, valid), gray(xb, valid))
    shift = float(np.hypot(dx, dy))
    meta.update(shift_px=shift, reg_response=float(resp))
    verdict = "accept"
    if shift > pc["reg_max_shift_px"]:
        if shift > pc["reg_repair_limit_px"] or resp < pc["reg_min_response"]:
            return {**meta, "verdict": "reject", "reason": "registration"}
        xb = shift_image(xb, dx, dy)
        valid &= shift_image(valid[None].astype("float32"), dx, dy)[0] > 0.5
        dx2, dy2, _ = phase_shift(gray(xa, valid), gray(xb, valid))
        meta["shift_after_px"] = float(np.hypot(dx2, dy2))
        if meta["shift_after_px"] > pc["reg_max_shift_px"]:
            return {**meta, "verdict": "reject", "reason": "registration_unrepaired"}
        verdict = "repair"
        meta["steps"].append("translation_correction")
    if a["modality"] == "optical" and pc["match_histograms"]:
        xb = match_histograms(xb, xa, channel_axis=0).astype("float32")
        meta["steps"].append("histogram_match")
    d = ensure(f"{cfg['paths']['prepared']}/{pid}")
    prof = {"driver": "GTiff", "height": h, "width": w, "crs": crs, "transform": transform, "compress": "deflate"}
    for name, arr in (("before", xa), ("after", xb)):
        with rasterio.open(d / f"{name}.tif", "w", count=arr.shape[0], dtype="float32", **prof) as f:
            f.write(arr.astype("float32"))
    with rasterio.open(d / "valid.tif", "w", count=1, dtype="uint8", **prof) as f:
        f.write(valid.astype("uint8")[None])
    lon = np.mean(cfg["aoi"]["bbox"][0::2])
    lat = np.mean(cfg["aoi"]["bbox"][1::2])
    cell = pc["dedupe_cell_deg"]
    meta["site_id"] = f"{round(lon / cell)}_{round(lat / cell)}"
    meta["verdict"] = verdict
    dump_json(meta, d / "pair.json")
    return meta


def run(cfg):
    cat = pd.read_csv(cfg["paths"]["catalog"])
    out = []
    for cand in build_candidates(cat, cfg):
        m = verify_and_prepare(cand, cfg)
        log.info("%s %s %s", m["pair_id"], m["verdict"], m.get("reason", ""))
        out.append({k: v for k, v in m.items() if k not in ("transform", "steps")})
    df = pd.DataFrame(out)
    df.to_csv(cfg["paths"]["pairs"], index=False)
    return df


def accepted_dirs(cfg, only=None):
    df = pd.read_csv(cfg["paths"]["pairs"])
    df = df[df["verdict"].isin(["accept", "repair"])]
    if only:
        df = df[df["pair_id"].isin(only)]
    return [Path(cfg["paths"]["prepared"]) / p for p in df["pair_id"]]

### `model.py`

In [ ]:
%%writefile model.py
import torch
import torch.nn as nn


def block(i, o):
    return nn.Sequential(
        nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
    )


class SiamUNet(nn.Module):
    def __init__(self, in_ch, widths):
        super().__init__()
        chans = [in_ch] + list(widths)
        n = len(widths)
        self.enc = nn.ModuleList([block(chans[i], chans[i + 1]) for i in range(n)])
        self.pool = nn.MaxPool2d(2)
        self.up = nn.ModuleList([nn.ConvTranspose2d(widths[i], widths[i - 1], 2, stride=2) for i in range(n - 1, 0, -1)])
        self.dec = nn.ModuleList([block(widths[i - 1] * 2, widths[i - 1]) for i in range(n - 1, 0, -1)])
        self.head = nn.Conv2d(widths[0], 1, 1)
        self.stride = 2 ** (n - 1)

    def encode(self, x):
        feats = []
        for i, e in enumerate(self.enc):
            x = e(x if i == 0 else self.pool(x))
            feats.append(x)
        return feats

    def forward(self, a, b):
        fa, fb = self.encode(a), self.encode(b)
        d = [(x - y).abs() for x, y in zip(fa, fb)]
        x = d[-1]
        for k, (u, dec) in enumerate(zip(self.up, self.dec)):
            x = dec(torch.cat([u(x), d[len(d) - 2 - k]], 1))
        return self.head(x)

### `train.py`

In [ ]:
%%writefile train.py
import random
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from common import ensure, get_logger
from model import SiamUNet

log = get_logger("train")


class PairFolder(Dataset):
    def __init__(self, root, split, dirs, patch, augment):
        base = Path(root) / split
        self.a = sorted((base / dirs["a"]).iterdir())
        self.b = base / dirs["b"]
        self.l = base / dirs["label"]
        self.patch = patch
        self.augment = augment

    def __len__(self):
        return len(self.a)

    def read(self, p, gray=False):
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE if gray else cv2.IMREAD_COLOR)
        return img if gray else img[:, :, ::-1]

    def __getitem__(self, i):
        pa = self.a[i]
        xa, xb, y = self.read(pa), self.read(self.b / pa.name), self.read(self.l / pa.name, True)
        h, w = y.shape
        py = random.randint(0, max(h - self.patch, 0))
        px = random.randint(0, max(w - self.patch, 0))
        sl = (slice(py, py + self.patch), slice(px, px + self.patch))
        xa, xb, y = xa[sl], xb[sl], y[sl]
        if self.augment:
            k = random.randint(0, 3)
            xa, xb, y = (np.rot90(t, k) for t in (xa, xb, y))
            if random.random() < 0.5:
                xa, xb, y = (t[:, ::-1] for t in (xa, xb, y))
        f = lambda t: torch.from_numpy(np.ascontiguousarray(t)).permute(2, 0, 1).float() / 255.0
        return f(xa), f(xb), torch.from_numpy((np.ascontiguousarray(y) > 127).astype("float32"))[None]


def dice_loss(logit, y):
    p = torch.sigmoid(logit)
    inter = (p * y).sum((1, 2, 3))
    return (1 - (2 * inter + 1) / (p.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1)).mean()


def evaluate(net, loader, dev):
    tp = fp = fn = 0
    net.eval()
    with torch.no_grad():
        for a, b, y in loader:
            p = (torch.sigmoid(net(a.to(dev), b.to(dev))) > 0.5).cpu()
            tp += int((p & (y > 0.5)).sum())
            fp += int((p & (y <= 0.5)).sum())
            fn += int((~p & (y > 0.5)).sum())
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return 2 * prec * rec / max(prec + rec, 1e-9), prec, rec


def run(cfg):
    t = cfg["train"]
    random.seed(t["seed"])
    torch.manual_seed(t["seed"])
    dev = torch.device(t["device"])
    tr = PairFolder(t["root"], t["splits"]["train"], t["dirs"], t["patch"], True)
    va = PairFolder(t["root"], t["splits"]["val"], t["dirs"], t["patch"], False)
    dl = DataLoader(tr, batch_size=t["batch"], shuffle=True, num_workers=t["workers"], drop_last=True)
    vl = DataLoader(va, batch_size=t["batch"], num_workers=t["workers"])
    net = SiamUNet(cfg["model"]["in_channels"], cfg["model"]["widths"]).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=t["lr"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, t["epochs"])
    pw = torch.tensor(t["pos_weight"], device=dev)
    best = -1
    ensure(Path(t["out"]).parent)
    for ep in range(t["epochs"]):
        net.train()
        tot = 0
        for a, b, y in dl:
            a, b, y = a.to(dev), b.to(dev), y.to(dev)
            out = net(a, b)
            loss = F.binary_cross_entropy_with_logits(out, y, pos_weight=pw) + t["dice_weight"] * dice_loss(out, y)
            opt.zero_grad()
            loss.backward()
            opt.step()
            tot += loss.item()
        sched.step()
        f1, p, r = evaluate(net, vl, dev)
        log.info("epoch %d loss %.4f val f1 %.4f p %.4f r %.4f", ep, tot / len(dl), f1, p, r)
        if f1 > best:
            best = f1
            torch.save({"state": net.state_dict(), "model": cfg["model"], "f1": f1}, t["out"])
    log.info("best val f1 %.4f -> %s", best, t["out"])

### `detect.py`

In [ ]:
%%writefile detect.py
import cv2
import numpy as np
import rasterio
from rasterio.features import shapes
from pyproj import Transformer
from shapely.geometry import mapping, shape
from shapely.ops import transform as shp_transform
from common import dump_json, get_logger, load_json
from pathlib import Path

log = get_logger("detect")


def load_pair(d):
    d = Path(d)
    with rasterio.open(d / "before.tif") as f:
        a, tf, crs = f.read(), f.transform, f.crs
    with rasterio.open(d / "after.tif") as f:
        b = f.read()
    with rasterio.open(d / "valid.tif") as f:
        v = f.read(1) > 0
    return a, b, v, tf, crs, load_json(d / "pair.json")


def robust_z(x, valid):
    vals = x[valid]
    med = np.median(vals)
    mad = np.median(np.abs(vals - med)) * 1.4826 + 1e-9
    return (x - med) / mad


def classical(a, b, valid, modality, cfg):
    dc = cfg["detect"]
    idx = [0] if modality == "sar" else dc["bands"]
    diff = b[idx] - a[idx]
    mag = np.sqrt((diff ** 2).mean(0)).astype("float32")
    mag = cv2.GaussianBlur(mag, (0, 0), dc["blur_sigma"][modality])
    z = robust_z(mag, valid)
    th = dc["threshold"]
    if th["type"] == "otsu":
        u8 = np.clip(255 * mag / max(mag[valid].max(), 1e-9), 0, 255).astype("uint8")
        t, _ = cv2.threshold(u8[valid].reshape(-1, 1), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        score = u8.astype("float32") / 255.0
        mask = u8 > t * th["value"]
    else:
        mask = z > th["value"]
        score = 1.0 / (1.0 + np.exp(-(z - th["value"])))
    return score.astype("float32"), mask & valid


def pad_to(n, t, step):
    return t - n if n < t else (-(n - t)) % step


def learned(a, b, valid, cfg):
    import torch
    from model import SiamUNet
    lc = cfg["detect"]["learned"]
    dev = torch.device(lc["device"])
    ck = torch.load(lc["checkpoint"], map_location=dev)
    net = SiamUNet(ck["model"]["in_channels"], ck["model"]["widths"]).to(dev)
    net.load_state_dict(ck["state"])
    net.eval()
    idx = cfg["detect"]["bands"]
    xa, xb = a[idx], b[idx]
    t, ov = lc["tile"], lc["overlap"]
    step = t - ov
    c, h, w = xa.shape
    ph, pw = pad_to(h, t, step), pad_to(w, t, step)
    xa = np.pad(xa, ((0, 0), (0, ph), (0, pw)), mode="reflect")
    xb = np.pad(xb, ((0, 0), (0, ph), (0, pw)), mode="reflect")
    H, W = xa.shape[1:]
    acc, cnt = np.zeros((H, W), "float32"), np.zeros((H, W), "float32")
    with torch.no_grad():
        for y in range(0, H - t + 1, step):
            for x in range(0, W - t + 1, step):
                ta = torch.from_numpy(xa[None, :, y:y + t, x:x + t]).to(dev)
                tb = torch.from_numpy(xb[None, :, y:y + t, x:x + t]).to(dev)
                acc[y:y + t, x:x + t] += torch.sigmoid(net(ta, tb))[0, 0].cpu().numpy()
                cnt[y:y + t, x:x + t] += 1
    prob = (acc / np.maximum(cnt, 1))[:h, :w]
    return prob, (prob > lc["threshold"]) & valid


def clean(mask, cfg):
    dc = cfg["detect"]
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dc["morph_kernel"],) * 2)
    m = cv2.morphologyEx(mask.astype("uint8"), cv2.MORPH_OPEN, k)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    keep = np.zeros(n, bool)
    keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= dc["min_area_px"]
    return keep[lab].astype("uint8")


def regionize(mask, prob, tf, crs):
    n, lab = cv2.connectedComponents(mask, connectivity=8)
    lab = lab.astype("int32")
    counts = np.bincount(lab.ravel(), minlength=n)
    sums = np.bincount(lab.ravel(), weights=prob.ravel(), minlength=n)
    to_ll = Transformer.from_crs(crs, "EPSG:4326", always_xy=True).transform
    inv = ~tf
    out = []
    for geom, val in shapes(lab, mask=lab > 0, transform=tf, connectivity=8):
        k = int(val)
        poly = shape(geom)
        ext = [list(inv * (x, y)) for x, y in poly.exterior.coords]
        px = np.array(ext)
        out.append({
            "id": k,
            "area_m2": poly.area,
            "area_px": int(counts[k]),
            "mean_prob": float(sums[k] / counts[k]),
            "bbox_px": [int(px[:, 0].min()), int(px[:, 1].min()), int(np.ceil(px[:, 0].max())), int(np.ceil(px[:, 1].max()))],
            "polygon_px": ext,
            "geometry_lonlat": mapping(shp_transform(to_ll, poly)),
        })
    return lab, out


def write_tif(path, arr, tf, crs, dtype):
    with rasterio.open(path, "w", driver="GTiff", height=arr.shape[0], width=arr.shape[1], count=1,
                       dtype=dtype, crs=crs, transform=tf, compress="deflate") as f:
        f.write(arr.astype(dtype)[None])


def process(d, cfg):
    a, b, valid, tf, crs, meta = load_pair(d)
    method = cfg["detect"]["method_by_modality"][meta["modality"]]
    prob, mask = classical(a, b, valid, meta["modality"], cfg) if method == "classical" else learned(a, b, valid, cfg)
    mask = clean(mask, cfg)
    lab, regions = regionize(mask, prob, tf, crs)
    d = Path(d)
    write_tif(d / "prob.tif", prob, tf, crs, "float32")
    write_tif(d / "labels.tif", lab, tf, crs, "int32")
    dump_json({"method": method, "regions": regions}, d / "regions.json")
    log.info("%s: %d regions (%s)", meta["pair_id"], len(regions), method)
    return regions


def run(cfg, pair_dirs):
    return {str(d): process(d, cfg) for d in pair_dirs}

### `interpret.py`

In [ ]:
%%writefile interpret.py
import base64
import json
import random
from pathlib import Path
import anthropic
import cv2
import numpy as np
import rasterio
from common import dump_json, get_logger, load_json

log = get_logger("interpret")


def parse_json(text):
    i, j = text.find("{"), text.rfind("}")
    if i < 0 or j < 0:
        return None
    try:
        return json.loads(text[i:j + 1])
    except json.JSONDecodeError:
        return None


def call_vlm(client, model, system, content, max_tokens):
    r = client.messages.create(model=model, max_tokens=max_tokens, system=system,
                               messages=[{"role": "user", "content": content}])
    return "".join(b.text for b in r.content if b.type == "text")


def to_display(arr, modality, vc):
    if modality == "sar":
        g = arr[0]
        lo, hi = np.percentile(g, vc["display_pct"])
        g = np.clip((g - lo) / max(hi - lo, 1e-6), 0, 1)
        rgb = np.stack([g, g, g])
    else:
        rgb = arr[vc["rgb_idx"]]
    return (np.clip(rgb, 0, 1) * 255).astype("uint8").transpose(1, 2, 0)


def window(bbox, shape, vc):
    x0, y0, x1, y1 = bbox
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    half = max(x1 - x0, y1 - y0, vc["min_chip_px"]) * (1 + vc["context_pad"]) / 2
    h, w = shape
    return (max(int(cx - half), 0), max(int(cy - half), 0), min(int(cx + half), w), min(int(cy + half), h))


def chip(img, win, vc, poly=None):
    x0, y0, x1, y1 = win
    c = img[y0:y1, x0:x1].copy()
    scale = vc["chip_px"] / max(c.shape[:2])
    if poly is not None:
        pts = (np.array(poly) - [x0, y0]).astype("int32").reshape(-1, 1, 2)
        cv2.polylines(c, [pts], True, (255, 0, 0), 2)
    c = cv2.resize(c, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    ok, buf = cv2.imencode(".jpg", c[:, :, ::-1], [cv2.IMWRITE_JPEG_QUALITY, vc["jpeg_quality"]])
    return base64.standard_b64encode(buf.tobytes()).decode()


def img_block(b64):
    return {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": b64}}


def build_content(prompt, imgs):
    out = []
    for label, b64 in imgs:
        out += [{"type": "text", "text": label}, img_block(b64)]
    return out + [{"type": "text", "text": prompt}]


def contrast(prob, lab, region, ring):
    x0, y0, x1, y1 = region["bbox_px"]
    h, w = lab.shape
    wx0, wy0, wx1, wy1 = max(x0 - ring - 1, 0), max(y0 - ring - 1, 0), min(x1 + ring + 1, w), min(y1 + ring + 1, h)
    L, P = lab[wy0:wy1, wx0:wx1], prob[wy0:wy1, wx0:wx1]
    inside = L == region["id"]
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * ring + 1,) * 2)
    ringm = (cv2.dilate(inside.astype("uint8"), k) > 0) & (L == 0)
    if not ringm.any():
        return float(P[inside].mean())
    return float(P[inside].mean() - P[ringm].mean())


def evaluate(desc, swap, null, judge, evidence, ic):
    f = []
    if desc is None:
        return ["vlm_parse_failed"], "uncertain"
    if not desc.get("change_detected"):
        f.append("vlm_no_change")
    if swap is not None:
        exp = ic["inverse_types"].get(desc.get("change_type"), desc.get("change_type"))
        if not swap.get("change_detected") or swap.get("change_type") != exp:
            f.append("swap_inconsistent")
    if null is not None and null.get("change_detected"):
        f.append("null_hallucination")
    if evidence["mean_prob"] < ic["min_prob"] or evidence["contrast"] < ic["min_contrast"]:
        f.append("weak_pixel_evidence")
    if desc.get("nuisance") not in (None, "none"):
        f.append("nuisance:" + str(desc.get("nuisance")))
    if judge is not None and not judge.get("supported"):
        f.append("judge_rejected")
    if not f:
        return f, "verified"
    return f, "rejected" if any(x in ic["reject_on"] for x in f) else "uncertain"


def interpret_pair(d, cfg, client):
    vc, ic, pr = cfg["vlm"], cfg["interpret"], cfg["prompts"]
    d = Path(d)
    meta = load_json(d / "pair.json")
    regs = load_json(d / "regions.json")["regions"]
    with rasterio.open(d / "before.tif") as f:
        a, tf, crs = f.read(), f.transform, f.crs
    with rasterio.open(d / "after.tif") as f:
        b = f.read()
    with rasterio.open(d / "prob.tif") as f:
        prob = f.read(1)
    with rasterio.open(d / "labels.tif") as f:
        lab = f.read(1)
    ia, ib = to_display(a, meta["modality"], vc), to_display(b, meta["modality"], vc)
    regs.sort(key=lambda r: r["area_m2"] * r["mean_prob"], reverse=True)
    fmt = dict(modality=meta["modality"], gsd=meta["res_m"], types=ic["change_types"], nuisance=ic["nuisance_types"])
    feats = []
    for n, r in enumerate(regs[: ic["max_regions"]]):
        rng = random.Random(ic["seed"] * 100003 + r["id"])
        win = window(r["bbox_px"], lab.shape, vc)
        cb, ca = chip(ia, win, vc), chip(ib, win, vc)
        ov = chip(ib, win, vc, r["polygon_px"])
        ovb = chip(ia, win, vc, r["polygon_px"])
        bd, ad = meta["before_date"][:10], meta["after_date"][:10]
        ask = lambda x, y, dx, dy, xo, p=pr["describe"]: parse_json(call_vlm(
            client, vc["model"], pr["system"],
            build_content(p.format(before_date=dx, after_date=dy, **fmt),
                          [("BEFORE", x), ("AFTER", y), ("AFTER with region outlined", xo)]), vc["max_tokens"]))
        desc = ask(cb, ca, bd, ad, ov)
        swap = ask(ca, cb, ad, bd, ovb) if rng.random() < ic["swap_rate"] else None
        null = ask(cb, cb, bd, bd, ovb) if rng.random() < ic["null_rate"] else None
        judge = None
        if ic["judge"] and desc:
            claim = json.dumps({k: desc.get(k) for k in ("change_type", "object", "before_state", "after_state", "description")})
            judge = parse_json(call_vlm(
                client, vc["judge_model"], pr["system"],
                build_content(pr["judge"].format(before_date=bd, after_date=ad, claim=claim),
                              [("BEFORE", cb), ("AFTER", ca), ("AFTER with region outlined", ov)]), vc["max_tokens"]))
        evidence = {"mean_prob": r["mean_prob"], "contrast": contrast(prob, lab, r, ic["ring_px"])}
        fails, status = evaluate(desc, swap, null, judge, evidence, ic)
        feats.append({
            "type": "Feature",
            "geometry": r["geometry_lonlat"],
            "properties": {"region_id": r["id"], "area_m2": r["area_m2"], **evidence, "status": status, "failures": fails,
                           "description": desc, "swap_check": swap, "null_check": null, "judge": judge},
        })
        log.info("%s region %s -> %s %s", meta["pair_id"], r["id"], status, fails)
    summary = None
    if ic["summarize"]:
        keep = [{"type": x["properties"]["description"].get("change_type"), "text": x["properties"]["description"].get("description"),
                 "status": x["properties"]["status"]} for x in feats if x["properties"]["status"] != "rejected" and x["properties"]["description"]]
        if keep:
            summary = call_vlm(client, vc["model"], pr["system"], [{"type": "text", "text": pr["summary"].format(
                before_date=meta["before_date"][:10], after_date=meta["after_date"][:10], findings=json.dumps(keep))}], vc["max_tokens"])
    rep = Path(cfg["paths"]["reports"]) / meta["pair_id"]
    rep.mkdir(parents=True, exist_ok=True)
    counts = {s: sum(1 for x in feats if x["properties"]["status"] == s) for s in ("verified", "uncertain", "rejected")}
    dump_json({"type": "FeatureCollection", "features": feats,
               "pair": {k: meta[k] for k in ("pair_id", "source", "modality", "before_id", "after_id", "before_date", "after_date", "gap_days", "license")},
               "counts": counts, "summary": summary}, rep / "report.geojson")
    return counts


def run(cfg, pair_dirs):
    client = anthropic.Anthropic()
    return {str(d): interpret_pair(d, cfg, client) for d in pair_dirs}

### `export_pair.py`

In [ ]:
%%writefile export_pair.py
import argparse
import shutil
from pathlib import Path
import cv2
import numpy as np
import rasterio
from common import ensure, load_config, load_json
from interpret import to_display
from pairing import accepted_dirs


def save_jpeg(path, img, ec, quality):
    s = ec["preview_px"] / max(img.shape[:2])
    if s < 1:
        img = cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    cv2.imwrite(str(path), img[:, :, ::-1], [cv2.IMWRITE_JPEG_QUALITY, quality])
    return s


def export(d, cfg):
    ec, vc = cfg["export"], cfg["vlm"]
    d = Path(d)
    meta = load_json(d / "pair.json")
    out = ensure(Path(ec["out_dir"]) / d.name)
    for name in ec["files"]:
        if (d / name).exists():
            shutil.copy2(d / name, out / name)
    with rasterio.open(d / "before.tif") as f:
        a = f.read()
    with rasterio.open(d / "after.tif") as f:
        b = f.read()
    ia, ib = to_display(a, meta["modality"], vc), to_display(b, meta["modality"], vc)
    save_jpeg(out / "before.jpg", ia, ec, vc["jpeg_quality"])
    s = save_jpeg(out / "after.jpg", ib, ec, vc["jpeg_quality"])
    over = ib.copy()
    for r in load_json(d / "regions.json")["regions"]:
        pts = np.array(r["polygon_px"]).astype("int32").reshape(-1, 1, 2)
        cv2.polylines(over, [pts], True, (255, 0, 0), 2)
    save_jpeg(out / "changes.jpg", over, ec, vc["jpeg_quality"])
    return out


def run(cfg, pair_dirs):
    return [export(d, cfg) for d in pair_dirs]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--pair", nargs="*")
    args = ap.parse_args()
    cfg = load_config(args.config)
    for out in run(cfg, accepted_dirs(cfg, args.pair)):
        print(out)


if __name__ == "__main__":
    main()

## Run

In [ ]:
import importlib, sys, os
sys.path.insert(0, os.getcwd())
import pandas as pd
import common, discover, pairing, detect, interpret, export_pair
for m in (common, discover, pairing, detect, interpret, export_pair):
    importlib.reload(m)
cfg = common.load_config('config.yaml')

### 1. Discover imagery

In [ ]:
catalog = discover.run(cfg)
catalog.head()

### 2. Select, verify and prepare pairs

In [ ]:
pairs = pairing.run(cfg)
pairs

In [ ]:
dirs = pairing.accepted_dirs(cfg)
len(dirs)

### 3. Detect and localize changes

In [ ]:
regions = detect.run(cfg, dirs)
{k: len(v) for k, v in regions.items()}

### 4. Export pairs and previews

In [ ]:
export_pair.run(cfg, dirs)

### 5. VLM description and verification

In [ ]:
counts = interpret.run(cfg, dirs)
counts

### 6. Review results

In [ ]:
import json
from pathlib import Path
rows = []
for d in dirs:
    rep = json.load(open(Path(cfg['paths']['reports']) / d.name / 'report.geojson'))
    for f in rep['features']:
        p = f['properties']
        desc = p['description'] or {}
        rows.append({'pair': d.name, 'region': p['region_id'], 'area_m2': round(p['area_m2']), 'status': p['status'],
                     'type': desc.get('change_type'), 'description': desc.get('description'), 'failures': ', '.join(p['failures'])})
pd.DataFrame(rows)

### Optional: train the learned detector

Needs a LEVIR-CD style folder at the path set in `config.yaml`.

In [ ]:
import train
importlib.reload(train)
train.run(cfg)